# QSE — end-to-end walkthrough

This notebook runs the full QSE loop in one place: it drives the **C++ backtest
engine via `subprocess`**, loads the equity / trade-log output it writes, and
renders an **inline performance tearsheet** (metrics + charts) — the same
`scripts/analysis/tearsheet.py` used to build the PDF report.

**Reproducible from a fresh clone:**

```bash
python3 -m venv venv
./venv/bin/pip install -r requirements-notebooks.txt
./venv/bin/jupyter nbconvert --to notebook --execute notebooks/qse_walkthrough.ipynb
```

If the C++ toolchain isn't available in the execution environment, the run-engine
step **falls back to a committed sample run** (produced by this same engine) so
the notebook always completes.

## 0 · Setup

In [ ]:
import subprocess, sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline

# Find the repo root from wherever nbconvert is executed (repo root or notebooks/).
REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "CMakeLists.txt").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
assert (REPO_ROOT / "CMakeLists.txt").exists(), "run this notebook from inside the QSE repo"

sys.path.insert(0, str(REPO_ROOT / "scripts" / "analysis"))
import tearsheet as ts  # the project's own metric library

print("repo root:", REPO_ROOT)

## 1 · Run the C++ engine (via subprocess)

`strategy_engine` runs an SMA 20/50 crossover backtest over the bundled AAPL
minute ticks (`data/raw_ticks_AAPL.csv`) and writes `equity_curve.csv` /
`tradelog.csv`. We build it if needed, then run it — and fall back to the
committed sample run if the build can't happen here.

In [ ]:
ENGINE = REPO_ROOT / "build" / "strategy_engine"
SAMPLE = REPO_ROOT / "notebooks" / "sample"


def ensure_engine() -> bool:
    """Return True if build/strategy_engine exists, building it if we can."""
    if ENGINE.exists():
        return True
    print("strategy_engine not built - attempting a cmake build ...")
    try:
        subprocess.run(
            ["cmake", "-S", str(REPO_ROOT), "-B", str(REPO_ROOT / "build"),
             "-DCMAKE_BUILD_TYPE=Release"],
            cwd=REPO_ROOT, check=True, capture_output=True, timeout=1800,
        )
        subprocess.run(
            ["cmake", "--build", str(REPO_ROOT / "build"),
             "--target", "strategy_engine", "-j"],
            cwd=REPO_ROOT, check=True, capture_output=True, timeout=1800,
        )
    except Exception as exc:  # no compiler / missing deps in this environment
        print(f"  build unavailable ({type(exc).__name__}) - will use the sample run")
    return ENGINE.exists()


try:
    if not ensure_engine():
        raise RuntimeError("engine binary unavailable")
    proc = subprocess.run(
        [str(ENGINE)], cwd=REPO_ROOT, check=True, capture_output=True,
        text=True, timeout=600,
    )
    print("\n".join(proc.stdout.strip().splitlines()[-3:]))
    EQUITY_PATH = REPO_ROOT / "equity_curve.csv"
    TRADELOG_PATH = REPO_ROOT / "tradelog.csv"
    SOURCE = "freshly run C++ engine"
except Exception as exc:
    print(f"falling back to the committed sample run ({type(exc).__name__}: {exc})")
    EQUITY_PATH, TRADELOG_PATH = SAMPLE / "equity_curve.csv", SAMPLE / "tradelog.csv"
    SOURCE = "committed sample run"

assert EQUITY_PATH.exists() and TRADELOG_PATH.exists(), "no engine output to load"
print("results from:", SOURCE)

## 2 · Load the results

In [ ]:
equity = ts.load_equity(str(EQUITY_PATH))      # timestamp,equity -> datetime series
trades = ts.load_tradelog(str(TRADELOG_PATH))   # per-fill blotter
daily = ts.to_daily(equity)                     # last obs per calendar day

print(f"{len(equity):,} equity points  ({equity.index[0].date()} -> {equity.index[-1].date()})")
print(f"{len(trades):,} fills  |  {len(daily)} daily observations")
equity.head()

## 3 · Inline tearsheet — metrics

Computed with the project's own `tearsheet` functions on the daily-resampled
curve, so the numbers match the PDF report the CLI produces.

In [ ]:
rets = ts.daily_returns(daily)
metrics = {
    "Total return": f"{equity.iloc[-1] / equity.iloc[0] - 1:.2%}",
    "CAGR": f"{ts.cagr(daily):.2%}",
    "Annualized Sharpe": f"{ts.annualized_sharpe(rets):.2f}",
    "Max drawdown": f"{ts.max_drawdown(daily):.2%}",
    "Calmar": f"{ts.calmar_ratio(daily):.2f}",
    "Annualized turnover": f"{ts.annualized_turnover(trades, daily):.2f}x",
    "Fills": f"{len(trades):,}",
}
pd.Series(metrics, name="value").to_frame()

## 4 · Inline tearsheet — charts

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 9), sharex=True)

axes[0].plot(daily.index, daily.to_numpy(), color="tab:blue", lw=1.4)
axes[0].set_title("Equity curve")
axes[0].set_ylabel("portfolio value")

dd = (daily - daily.cummax()) / daily.cummax()
axes[1].fill_between(dd.index, dd.to_numpy(), 0, color="tab:red", alpha=0.4)
axes[1].set_title("Drawdown (underwater)")
axes[1].set_ylabel("drawdown")

window = min(20, max(2, len(rets) // 3))
rs = ts.rolling_sharpe(rets, window=window)
axes[2].plot(rs.index, rs.to_numpy(), color="tab:green", lw=1.4)
axes[2].axhline(0, color="gray", lw=0.8)
axes[2].set_title(f"Rolling Sharpe ({window}-day window)")
axes[2].set_ylabel("annualized Sharpe")

fig.suptitle(f"QSE SMA 20/50 backtest - {SOURCE}", fontsize=13)
fig.tight_layout()
plt.show()

## Where to go next

- **The flagship result** — [A/B slippage audit](../docs/research/microstructure/ab_audit_summary.md):
  the same signals under naive vs full-depth fills.
- **The research track** — [stat arb](../docs/research/statarb/README.md) ·
  [validation / CPCV+DSR](../docs/research/validation/README.md) ·
  [regime](../docs/research/regime/README.md) ·
  [execution](../docs/research/execution/README.md) ·
  [meta-labeling](../docs/research/meta/README.md) ·
  [portfolio / HRP](../docs/research/portfolio/README.md).
- **The whitepaper** — [PROJECT_PHASES.md](../docs/PROJECT_PHASES.md) ·
  **the log** — [TASK_BREAKDOWN.md](../docs/TASK_BREAKDOWN.md).